In [1]:
!pip install huggingface_hub timm --quiet

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [14]:
import glob

BASE = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

# Collect all image paths from all 12 folders into one dict
image_paths = {}
for folder in glob.glob(f"{BASE}/images_*"):
    for img_path in glob.glob(f"{folder}/images/*.png"):
        filename = os.path.basename(img_path)
        image_paths[filename] = img_path

print(f"Total images found: {len(image_paths)}")

LABELS_CSV = f"{BASE}/Data_Entry_2017.csv"

LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax", "Consolidation",
    "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
IMG_SIZE = 224

Total images found: 112120


In [9]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/organizations/nih-chest-xrays/data/BBox_List_2017.csv
/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv


In [15]:
df = pd.read_csv(LABELS_CSV)

for label in LABELS:
    df[label] = df["Finding Labels"].apply(lambda x: 1.0 if label in x else 0.0)

# Map each image to its full path
df["full_path"] = df["Image Index"].map(image_paths)
df = df[df["full_path"].notna()].reset_index(drop=True)
print(f"Total valid images: {len(df)}")

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Total valid images: 112120
Train: 100908 | Val: 11212


In [13]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/organizations/nih-chest-xrays/data"):
    for d in dirs:
        print(os.path.join(root, d))
    break

/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_003
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_012
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_009
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_008
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_007
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_010
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_002
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_011
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_001
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_005
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_004
/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_006


In [17]:
class XrayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["full_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        labels = torch.tensor(row[LABELS].values.astype(float), dtype=torch.float32)
        return image, labels

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = XrayDataset(train_df, train_transform)
val_dataset = XrayDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print("Dataloaders ready")

Dataloaders ready


In [18]:
model = models.densenet121(weights="IMAGENET1K_V1")
model.classifier = nn.Linear(1024, 14)
model = model.to(DEVICE)

# Handle class imbalance with pos_weight
pos_counts = train_df[LABELS].sum()
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor((neg_counts / pos_counts).values, dtype=torch.float32).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
print("Model ready")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 178MB/s]


Model ready


In [19]:
best_auc = 0.0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # --- Validate ---
    model.eval()
    all_probs, all_labels = [], []
    val_loss = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # AUC per class
    aucs = []
    for i, label in enumerate(LABELS):
        if all_labels[:, i].sum() > 0:
            auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
            aucs.append(auc)
    mean_auc = np.mean(aucs)

    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | "
          f"Mean AUC: {mean_auc:.4f}")

    # Save best model
    if mean_auc > best_auc:
        best_auc = mean_auc
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print(f"  Best model saved (AUC: {best_auc:.4f})")

print(f"\nTraining complete. Best AUC: {best_auc:.4f}")

Epoch 1/10 | Train Loss: 1.0976 | Val Loss: 1.0114 | Mean AUC: 0.8002
  Best model saved (AUC: 0.8002)
Epoch 2/10 | Train Loss: 0.9966 | Val Loss: 0.9815 | Mean AUC: 0.8083
  Best model saved (AUC: 0.8083)
Epoch 3/10 | Train Loss: 0.9598 | Val Loss: 0.9474 | Mean AUC: 0.8252
  Best model saved (AUC: 0.8252)
Epoch 4/10 | Train Loss: 0.8841 | Val Loss: 0.9223 | Mean AUC: 0.8326
  Best model saved (AUC: 0.8326)
Epoch 5/10 | Train Loss: 0.8581 | Val Loss: 0.9177 | Mean AUC: 0.8370
  Best model saved (AUC: 0.8370)
Epoch 6/10 | Train Loss: 0.8424 | Val Loss: 0.9181 | Mean AUC: 0.8389
  Best model saved (AUC: 0.8389)
Epoch 7/10 | Train Loss: 0.8235 | Val Loss: 0.9164 | Mean AUC: 0.8391
  Best model saved (AUC: 0.8391)
Epoch 8/10 | Train Loss: 0.8222 | Val Loss: 0.9193 | Mean AUC: 0.8390
Epoch 9/10 | Train Loss: 0.8199 | Val Loss: 0.9223 | Mean AUC: 0.8393
  Best model saved (AUC: 0.8393)
Epoch 10/10 | Train Loss: 0.8169 | Val Loss: 0.9187 | Mean AUC: 0.8393
  Best model saved (AUC: 0.8393)

T

In [20]:
from huggingface_hub import HfApi
import getpass

api = HfApi()

# Create repo (change YOUR_USERNAME to your HuggingFace username)
repo_id = "ManasviSaireddy/chestxray-densenet121"

api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_file(
    path_or_fileobj="/kaggle/working/best_model.pth",
    path_in_repo="best_model.pth",
    repo_id=repo_id,
)
print(f"Weights uploaded to: https://huggingface.co/{repo_id}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Weights uploaded to: https://huggingface.co/ManasviSaireddy/chestxray-densenet121


In [22]:
import os

# Search for test_list.txt
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "test_list" in file.lower():
            print(os.path.join(root, file))

/kaggle/input/datasets/organizations/nih-chest-xrays/data/test_list.txt


In [23]:
# Official NIH test split
test_list_path = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/test_list.txt"
with open(test_list_path) as f:
    test_images = set(f.read().splitlines())

test_df = df[df["Image Index"].isin(test_images)].reset_index(drop=True)
print(f"Official test set size: {len(test_df)}")

# Load best model
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()

test_dataset = XrayDataset(test_df, val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

print("\nPer-class AUC on official NIH test set:")
print("-" * 40)
aucs = {}
for i, label in enumerate(LABELS):
    if all_labels[:, i].sum() > 0:
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        aucs[label] = round(auc, 4)
        print(f"{label:<25} AUC: {auc:.4f}")

print("-" * 40)
print(f"Mean AUC: {np.mean(list(aucs.values())):.4f}")

Official test set size: 25596

Per-class AUC on official NIH test set:
----------------------------------------
Atelectasis               AUC: 0.7785
Cardiomegaly              AUC: 0.9258
Effusion                  AUC: 0.8366
Infiltration              AUC: 0.7208
Mass                      AUC: 0.8573
Nodule                    AUC: 0.7835
Pneumonia                 AUC: 0.7912
Pneumothorax              AUC: 0.8987
Consolidation             AUC: 0.7764
Edema                     AUC: 0.8837
Emphysema                 AUC: 0.9500
Fibrosis                  AUC: 0.8893
Pleural_Thickening        AUC: 0.8170
Hernia                    AUC: 0.9876
----------------------------------------
Mean AUC: 0.8497


In [24]:
print("\nPer-class AUC scores:")
print("-" * 35)
for i, label in enumerate(LABELS):
    if all_labels[:, i].sum() > 0:
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        status = "✓ Good" if auc >= 0.75 else "⚠ Needs work"
        print(f"{label:<22} {auc:.4f}  {status}")


Per-class AUC scores:
-----------------------------------
Atelectasis            0.7785  ✓ Good
Cardiomegaly           0.9258  ✓ Good
Effusion               0.8366  ✓ Good
Infiltration           0.7208  ⚠ Needs work
Mass                   0.8573  ✓ Good
Nodule                 0.7835  ✓ Good
Pneumonia              0.7912  ✓ Good
Pneumothorax           0.8987  ✓ Good
Consolidation          0.7764  ✓ Good
Edema                  0.8837  ✓ Good
Emphysema              0.9500  ✓ Good
Fibrosis               0.8893  ✓ Good
Pleural_Thickening     0.8170  ✓ Good
Hernia                 0.9876  ✓ Good


In [26]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

BASE = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"

# Build image index from all subfolders
image_paths = {}
for folder in os.listdir(BASE):
    img_dir = os.path.join(BASE, folder, "images")
    if os.path.exists(img_dir):
        for fname in os.listdir(img_dir):
            image_paths[fname] = os.path.join(img_dir, fname)

print(f"Total images found: {len(image_paths)}")

# Load CSVs
labels_df = pd.read_csv(os.path.join(BASE, "Data_Entry_2017.csv"))
test_list = pd.read_csv(os.path.join(BASE, "test_list.txt"), header=None, names=["Image Index"])

# Merge
test_df = test_list.merge(labels_df, on="Image Index")
for label in LABELS:
    test_df[label] = test_df["Finding Labels"].apply(lambda x: 1.0 if label in x else 0.0)

test_df["full_path"] = test_df["Image Index"].map(image_paths)
test_df = test_df[test_df["full_path"].notna()].reset_index(drop=True)
print(f"Test images found: {len(test_df)}")

Total images found: 112120
Test images found: 25596


In [27]:
class TestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["full_path"]).convert("RGB")
        image = self.transform(image)
        labels = torch.tensor(row[LABELS].values.astype(float), dtype=torch.float32)
        return image, labels

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_dataset = TestDataset(test_df, val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

# Load best weights
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()

# Run predictions
all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = torch.sigmoid(model(images)).cpu().numpy()
        all_probs.append(outputs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

# Per class AUC
print("\nPer-class AUC on official test set:")
print("-" * 40)
aucs = {}
for i, label in enumerate(LABELS):
    if all_labels[:, i].sum() > 0:
        auc = roc_auc_score(all_labels[:, i], all_probs[:, i])
        aucs[label] = round(auc, 4)
        print(f"{label:<25} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(list(aucs.values())):.4f}")


Per-class AUC on official test set:
----------------------------------------
Atelectasis               AUC: 0.7785
Cardiomegaly              AUC: 0.9258
Effusion                  AUC: 0.8366
Infiltration              AUC: 0.7208
Mass                      AUC: 0.8573
Nodule                    AUC: 0.7835
Pneumonia                 AUC: 0.7912
Pneumothorax              AUC: 0.8987
Consolidation             AUC: 0.7764
Edema                     AUC: 0.8837
Emphysema                 AUC: 0.9500
Fibrosis                  AUC: 0.8893
Pleural_Thickening        AUC: 0.8170
Hernia                    AUC: 0.9876

Mean AUC: 0.8497


In [2]:
import pandas as pd
import os
import shutil

LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax", "Consolidation",
    "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"
]

# Find one confirmed image per disease
BASE = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
labels_df = pd.read_csv(os.path.join(BASE, "Data_Entry_2017.csv"))
test_list = pd.read_csv(os.path.join(BASE, "test_list.txt"), header=None, names=["Image Index"])
test_df = test_list.merge(labels_df, on="Image Index")

# Build image path index
image_paths = {}
for folder in os.listdir(BASE):
    img_dir = os.path.join(BASE, folder, "images")
    if os.path.exists(img_dir):
        for fname in os.listdir(img_dir):
            image_paths[fname] = os.path.join(img_dir, fname)

os.makedirs("/kaggle/working/test_samples", exist_ok=True)

# Save one confirmed image per disease
for label in LABELS:
    subset = test_df[test_df["Finding Labels"].str.contains(label, na=False)]
    pure = subset[subset["Finding Labels"] == label]
    if len(pure) == 0:
        pure = subset
    row = pure.iloc[0]
    src = image_paths.get(row["Image Index"])
    if src:
        dst = f"/kaggle/working/test_samples/{label}__{row['Image Index']}"
        shutil.copy(src, dst)
        print(f"Saved: {label} -> {row['Image Index']}")

Saved: Atelectasis -> 00000032_054.png
Saved: Cardiomegaly -> 00000013_045.png
Saved: Effusion -> 00000013_021.png
Saved: Infiltration -> 00000013_007.png
Saved: Mass -> 00000013_024.png
Saved: Nodule -> 00000344_004.png
Saved: Pneumonia -> 00000193_019.png
Saved: Pneumothorax -> 00000013_011.png
Saved: Consolidation -> 00000032_016.png
Saved: Edema -> 00000032_024.png
Saved: Emphysema -> 00000013_041.png
Saved: Fibrosis -> 00000092_002.png
Saved: Pleural_Thickening -> 00000013_003.png
Saved: Hernia -> 00000003_000.png


In [3]:
import shutil
shutil.make_archive('test_samples_download','zip','/kaggle/working/test_samples')

'/kaggle/working/test_samples_download.zip'